In [ ]:
"""
SCRIPT 02
Membuat centroid desa/kelurahan untuk Provinsi Maluku
dan Maluku Utara.
Input:
    BATAS WILAYAH KELURAHAN-DESA 10K.zip
Output:
    Output/Desa_Centroid_Master.csv
"""
from pathlib import Path
import warnings
import geopandas as gpd
# ============================================================
# PATH
# ============================================================
PROJECT_ROOT = Path.cwd().parent  # run from the Script/ folder
DATA_DIR = PROJECT_ROOT / "Data Kementrian Koordinator dan FKTP"
OUTPUT_DIR = PROJECT_ROOT / "Output"
OUTPUT_DIR.mkdir(
    exist_ok=True
)
# ============================================================
# FILE SHAPEFILE
# ============================================================
ZIP = PROJECT_ROOT / (
    "BATAS WILAYAH KELURAHAN-DESA 10K.zip"
)
SHP = (
    f"/vsizip/{ZIP.as_posix()}/"
    "Batas_Wilayah_KelurahanDesa_10K_AR.shp"
)
# ============================================================
# OUTPUT
# ============================================================
OUT = OUTPUT_DIR / (
    "Desa_Centroid_Master.csv"
)
# ============================================================
# MAPPING KOLOM SHAPEFILE
# ============================================================
RENAME = {
    "KDEPUM":
        "KDDESA",
    "WADMKD":
        "NMDESA2",
    "WADMKC":
        "NMKEC",
    "WADMKK":
        "NMDATI2",
    "WADMPR":
        "NMPROP"
}
# ============================================================
# MAIN
# ============================================================
def main():
    # --------------------------------------------------------
    # 1. FILTER PROVINSI
    # --------------------------------------------------------
    where = (
        "WADMPR IN "
        "('Maluku', 'Maluku Utara')"
    )
    # --------------------------------------------------------
    # 2. BACA SHAPEFILE
    # --------------------------------------------------------
    print(
        "Membaca shapefile..."
    )
    gdf = gpd.read_file(
        SHP,
        columns=list(
            RENAME.keys()
        ),
        where=where
    )
    print()
    print(
        f"Desa terbaca: {len(gdf)}"
    )
    # --------------------------------------------------------
    # 3. CEK CRS
    # --------------------------------------------------------
    print()
    print(
        f"CRS shapefile: {gdf.crs}"
    )
    # --------------------------------------------------------
    # 4. HITUNG CENTROID
    # --------------------------------------------------------
    print()
    print(
        "Menghitung centroid..."
    )
    # Shapefile menggunakan sistem koordinat geografis
    # sehingga koordinat centroid dihitung dalam longitude
    # dan latitude.
    with warnings.catch_warnings():
        warnings.simplefilter(
            "ignore"
        )
        centroid = (
            gdf.geometry.centroid
        )
    # --------------------------------------------------------
    # 5. BUAT DATAFRAME OUTPUT
    # --------------------------------------------------------
    out = (
        gdf[
            list(
                RENAME.keys()
            )
        ]
        .rename(
            columns=RENAME
        )
        .copy()
    )
    # --------------------------------------------------------
    # 6. SIMPAN KOORDINAT CENTROID
    # --------------------------------------------------------
    out["Longitude"] = (
        centroid.x.round(6)
    )
    out["Latitude"] = (
        centroid.y.round(6)
    )
    # --------------------------------------------------------
    # 7. CEK KOORDINAT INVALID
    # --------------------------------------------------------
    bad = (
        out[
            [
                "Longitude",
                "Latitude"
            ]
        ]
        .isna()
        .any(

            axis=1
        )
        .sum()
    )
    # --------------------------------------------------------
    # 8. SIMPAN CSV
    # --------------------------------------------------------
    out.to_csv(
        OUT,
        index=False,
        encoding="utf-8"
    )
    # --------------------------------------------------------
    # 9. INFORMASI HASIL
    # --------------------------------------------------------
    print()
    print(

        "Jumlah desa per provinsi:"
    )
    print(
        out["NMPROP"]
        .value_counts()
        .to_string()
    )
    print()
    print(
        f"Centroid gagal: {bad}"
    )
    print()
    print(
        "Contoh data hasil:"
    )
    print(
        out.head()
        .to_string(
            index=False
        )
    )
    print()
    print(
        "Output tersimpan di:"
    )
    print(
        OUT
    )
# ============================================================
# RUN
# ============================================================
if __name__ == "__main__":
    main()